In [1]:
import sys

from hydra.utils import instantiate
from einops import rearrange
from omegaconf import OmegaConf
import torch
import os
import sys

sys.path.append(os.path.abspath("/svl/u/ravenh/lacwm/robot_world_models-raven-lam/projects/latent_action_models")
                )
# Register custom resolver to handle multiplication in OmegaConf interpolation
OmegaConf.register_new_resolver("mul", lambda x, y: float(x) * float(y))
OmegaConf.register_new_resolver("div", lambda a, b: float(a) / float(b))

In [2]:
complete_cfg = OmegaConf.load("/svl/u/ravenh/lacwm/robot_world_models-raven-lam/projects/latent_action_models/data/experiments_0908/ewm_agibot_egodex_droid_tfds_mt_xatten_cosmos_singlemlp/2026-03-27/01-48-47/.hydra/config.yaml")
complete_val_dataloader = instantiate(complete_cfg.val_data_loader)

/svl/u/ravenh/miniconda3/envs/robot_world_models/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
snapshot_dir = "/svl/u/ravenh/lacwm/robot_world_models-raven-lam/projects/latent_action_models/data/experiments_0908/ewm_agibot_egodex_droid_tfds_mt_xatten_cosmos_singlemlp/2026-03-27/01-48-47"
config_path = f"{snapshot_dir}/.hydra"
print(config_path)

cfg = OmegaConf.load(config_path + "/config.yaml")

/svl/u/ravenh/lacwm/robot_world_models-raven-lam/projects/latent_action_models/data/experiments_0908/ewm_agibot_egodex_droid_tfds_mt_xatten_cosmos_singlemlp/2026-03-27/01-48-47/.hydra


In [4]:
os.environ["COSMOS_HOME"] = "/svl/u/ravenh/lacwm/Cosmos"

In [5]:

model = instantiate(cfg.model)
model = model.cuda().eval()
snapshot = torch.load(f"{snapshot_dir}/snapshot.pt", map_location="cuda:0", weights_only=True)
model.load_state_dict(snapshot["model"])
val_dataloader = instantiate(cfg.val_data_loader)

In [6]:
val_dataloader[1].dataset.datasets.keys()

dict_keys(['EgoDex'])

In [7]:
droid_dataloader = complete_val_dataloader[2]
egodex_dataloader = complete_val_dataloader[1]
agibot_dataloader = complete_val_dataloader[0]

In [ ]:
droid_zs_bs = []
for i, droid_batch in enumerate(droid_dataloader):
    
    z = model._forward_action_encode( droid_batch["actions"].cuda(), 
                                    droid_batch["morphology_index"].cuda(),
                                    droid_batch["ee_action_dim"].cuda(), rgb_tokens=None ).detach().cpu().numpy()

    droid_zs_bs.append(z)
    if i > 100:
        break

In [ ]:


egodex_zs_bs = []
for i, egodex_batch in enumerate(egodex_dataloader):

    z = model._forward_action_encode( egodex_batch["actions"].cuda(), 
                                    egodex_batch["morphology_index"].cuda(),
                                    egodex_batch["ee_action_dim"].cuda(), rgb_tokens=None ).detach().cpu().numpy()
    egodex_zs_bs.append(z)
    if i > 100:
        break
    


In [14]:
import numpy as np

mds = agibot_dataloader.dataset                       # MultiDataset
ads = mds.datasets["Agibot"]                          # AgibotDataset
ads.all_task_episodes = [te for te in ads.all_task_episodes if te[0] == "327"]

# MultiDataset caches sub-dataset lengths at construction — recompute them
mds.dataset_lengths   = [len(d) for d in mds.datasets.values()]
mds.cumulative_lengths = np.cumsum(mds.dataset_lengths)

print(f"agibot episodes: {len(ads.all_task_episodes)}, multi total: {len(mds)}")
# expect: 60 / 60

agibot episodes: 60, multi total: 60


In [15]:
agibot_zs_bs = []
for i, agibot_batch in enumerate(agibot_dataloader):

    z = model._forward_action_encode( agibot_batch["actions"].cuda(), 
                                    agibot_batch["morphology_index"].cuda(),
                                    agibot_batch["ee_action_dim"].cuda(), rgb_tokens=None ).detach().cpu().numpy()
    agibot_zs_bs.append(z)
    if i > 100:
        break

Exception ignored in: <function _StatefulMultiProcessingDataLoaderIter.__del__ at 0x7a2c1ba49750>
Traceback (most recent call last):
  File "/svl/u/ravenh/miniconda3/envs/robot_world_models/lib/python3.10/site-packages/torchdata/stateful_dataloader/stateful_dataloader.py", line 1622, in __del__
    self._shutdown_workers()
  File "/svl/u/ravenh/miniconda3/envs/robot_world_models/lib/python3.10/site-packages/torchdata/stateful_dataloader/stateful_dataloader.py", line 1605, in _shutdown_workers
    if w.is_alive():
  File "/svl/u/ravenh/miniconda3/envs/robot_world_models/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _StatefulMultiProcessingDataLoaderIter.__del__ at 0x7a2c1ba49750>
Traceback (most recent call last):
  File "/svl/u/ravenh/miniconda3/envs/robot_world_models/lib/python3.10/site-packages/torchdata/statefu

In [ ]:
import mediapy as media
media.write_video("droid_0.mp4", rearrange(droid_batch['rgb'][0].cpu().numpy(), "t c h w -> t h w c"), fps=10)

In [ ]:
import numpy as np

In [16]:
droid_zs_bs_c = np.concatenate(droid_zs_bs, axis=0)
egodex_zs_bs_c = np.concatenate(egodex_zs_bs, axis=0)
agibot_zs_bs_c = np.concatenate(agibot_zs_bs, axis=0)

In [19]:
flatten_droid_zs_bs = rearrange(droid_zs_bs_c, "b t a m -> (b t a) m")
flatten_egodex_zs_bs = rearrange(egodex_zs_bs_c, "b t a m -> (b t a) m")
flatten_agibot_zs_bs = rearrange(agibot_zs_bs_c, "b t a m -> (b t a) m")

In [20]:
bs_embeddings = np.stack([flatten_droid_zs_bs, flatten_egodex_zs_bs, flatten_agibot_zs_bs], axis=0)
# bs_embeddings = np.stack([flatten_droid_zs_bs, flatten_egodex_zs_bs], axis=0)


In [21]:
import umap
import numpy as np
import plotly.express as px
import torch

def plot_umap_3d_interactive(embeddings, color_labels=None):
    """
    embeddings: torch.Tensor or np.ndarray of shape (S, B, M)
    color_labels: optional array of shape (S*B,) for coloring
    """
    # Convert to numpy
    if torch.is_tensor(embeddings):
        embeddings = embeddings.detach().cpu().numpy()

    S, B, M = embeddings.shape
    embeddings_flat = embeddings.reshape(S * B, M)

    # Default color labels = data source index
    if color_labels is None:
        color_labels = np.repeat(np.arange(S), B)

    # Run UMAP
    reducer = umap.UMAP(n_components=3, random_state=42)
    embeddings_3d = reducer.fit_transform(embeddings_flat)

    # Create interactive plot
    fig = px.scatter_3d(
        x=embeddings_3d[:, 0],
        y=embeddings_3d[:, 1],
        z=embeddings_3d[:, 2],
        color=color_labels.astype(str),  # plotly requires string or category
        labels={'color': 'Data Source'},
        title="Interactive 3D UMAP"
    )
    fig.update_traces(marker=dict(size=3, opacity=0.7))
    noaxis = dict(
        showline=False,
        zeroline=False,
        showticklabels=False,
        title='',
        showspikes=False,
    )
    fig.update_layout(
        margin=dict(l=0, r=0, b=0, t=30),
        scene=dict(
            xaxis=noaxis,
            yaxis=noaxis,
            zaxis=noaxis,
        )
    )
    fig.show()

In [ ]:
S, B, M = bs_embeddings.shape

In [ ]:
droid_me = model.morphology_tokens.weight[0]

In [ ]:
egodex_me = model.morphology_tokens.weight[2]

In [ ]:
plot_umap_3d_interactive(bs_embeddings, color_labels=np.repeat(np.arange(3), bs_embeddings.shape[1]))

In [ ]:
embed_pm = torch.stack([egodex_me, droid_me,droid_me ], dim=0).detach().cpu().numpy()